## WAP to produce movie recommendations.

In [1]:
from pyspark.sql import SparkSession
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator

In [2]:
spark = SparkSession.builder.getOrCreate()

24/04/04 06:11:33 WARN Utils: Your hostname, Asadullahs-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 192.168.0.114 instead (on interface en0)
24/04/04 06:11:33 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/04/04 06:11:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [ ]:
data = spark.read.csv('datasets/movie_ratings.csv', header=True, inferSchema=True)

In [ ]:
data.printSchema()
data.show(5)

root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- rating: double (nullable = true)
 |-- timestamp: timestamp (nullable = true)

+------+-------+------+-------------------+
|userId|movieId|rating|          timestamp|
+------+-------+------+-------------------+
|     1|      2|   3.5|2005-04-02 23:53:47|
|     1|     29|   3.5|2005-04-02 23:31:16|
|     1|     32|   3.5|2005-04-02 23:33:39|
|     1|     47|   3.5|2005-04-02 23:32:07|
|     1|     50|   3.5|2005-04-02 23:29:40|
+------+-------+------+-------------------+
only showing top 5 rows



In [ ]:
evaluator = RegressionEvaluator(predictionCol='prediction', labelCol='rating', metricName='mse')

In [ ]:
data = data.drop('timestamp')
train_data, test_data = data.randomSplit([0.8, 0.2], seed=42)

In [ ]:
als = ALS(maxIter=10, regParam=0.5, userCol='userId', itemCol='movieId', ratingCol='rating', coldStartStrategy='drop')
als_model = als.fit(train_data)
als_pred = als_model.transform(test_data)

In [ ]:
evaluator.evaluate(als_pred)

0.9957116801311653

In [ ]:
recommended_movie_df = als_model.recommendForAllUsers(3)
recommended_movie_df.show(10)

+------+---------------------------------------------------------------+
|userId|recommendations                                                |
+------+---------------------------------------------------------------+
|1     |[{126219, 6.3411317}, {125599, 5.5719476}, {114271, 5.5719476}]|
|6     |[{126219, 6.7395463}, {125599, 5.921879}, {114271, 5.921879}]  |
|12    |[{126219, 6.2000055}, {125599, 5.447845}, {114271, 5.447845}]  |
|13    |[{126219, 6.589937}, {125599, 5.7904496}, {114271, 5.7904496}] |
|16    |[{126219, 6.1974025}, {125599, 5.445632}, {114271, 5.445632}]  |
|22    |[{126219, 6.4893737}, {125599, 5.7021503}, {114271, 5.7021503}]|
|26    |[{126219, 6.3442116}, {125599, 5.574588}, {114271, 5.574588}]  |
|27    |[{126219, 6.6514277}, {125599, 5.844593}, {114271, 5.844593}]  |
|28    |[{126219, 4.8590226}, {125599, 4.269637}, {114271, 4.269637}]  |
|31    |[{126219, 5.948939}, {125599, 5.22725}, {114271, 5.22725}]     |
+------+-------------------------------------------

In [ ]:
user_subset = data.select('userId').distinct()
recommendations = als_model.recommendForUserSubset(user_subset, 3)
recommendations.show(10)

+------+---------------------------------------------------------------+
|userId|recommendations                                                |
+------+---------------------------------------------------------------+
|1     |[{126219, 6.3411317}, {125599, 5.5719476}, {114271, 5.5719476}]|
|6     |[{126219, 6.7395463}, {125599, 5.921879}, {114271, 5.921879}]  |
|12    |[{126219, 6.2000055}, {125599, 5.447845}, {114271, 5.447845}]  |
|13    |[{126219, 6.589937}, {125599, 5.7904496}, {114271, 5.7904496}] |
|16    |[{126219, 6.1974025}, {125599, 5.445632}, {114271, 5.445632}]  |
|22    |[{126219, 6.4893737}, {125599, 5.7021503}, {114271, 5.7021503}]|
|26    |[{126219, 6.3442116}, {125599, 5.574588}, {114271, 5.574588}]  |
|27    |[{126219, 6.6514277}, {125599, 5.844593}, {114271, 5.844593}]  |
|28    |[{126219, 4.8590226}, {125599, 4.269637}, {114271, 4.269637}]  |
|31    |[{126219, 5.948939}, {125599, 5.22725}, {114271, 5.22725}]     |
+------+-------------------------------------------